# Add metrics and plots without changing the framework
The functions in `gaussian/extensions.py` add a model-based metric, a shared-dataset metric, and a plot that can access the model, simulator, prior and observations. No functions are defined in this notebook. The `extended` preset in `settings.py` reuses the smoke output store so existing training is reused.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "settings.py").is_file() and (p / "nnpd").is_dir())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from settings import make_config
from nnpd import execute, plan, restore, load_experiment
from nnpd.results import runs

In [ ]:
config = make_config("extended")
experiment = load_experiment(config["application"])
paths = execute(config, experiment, settings_file=ROOT / "settings.py")

In [ ]:
context = restore(paths[0], experiment)
hook = experiment.metrics()["median_absolute_bias"]
hook.compute(context, context.dependencies(hook.needs))

In [ ]:
hook = experiment.metrics()["parameter_count"]
hook.compute(context, context.dependencies(hook.needs))

For a new shared dataset, register a `Product(builder, needs=(...), settings=...)`. Its builder receives a writer for arrays/JSON. List that product in any number of metric or plot dependencies. See `docs/EXTENDING.md` for the complete interface and cache-key contract.